# 会話文 → トリアージ 推定パイプライン (SBERT 版)

**目的**: `dataset/headache_emergency_calls.txt` の通話文から、`protocol.yaml` の遷移図を機械的に辿って
最終トリアージを推定する。

## パイプライン

```
[会話 (Dispatcher / Caller)] 
  → 発話分割（話者ラベル付き）
  → protocol を start_node から 1 ノードずつ進める
      ├ 質問ノード q について:
      │   1) q のテキストを SBERT 埋め込み → dispatcher 発話列との cos 類似度最大を選ぶ
      │   2) その発話直後の caller 発話 r を取得
      │   3) 各 choice の {質問+選択肢} テンプレを SBERT 埋め込みし、r と最大類似 → choice 決定
      │   4) 決まった choice で YAML を遷移
      └ 終端 / route_to_protocol / fallback で停止
  → 最終トリアージ
```

## 使用モデル

- `intfloat/multilingual-e5-small` (384 次元, 多言語 SBERT)
- 旧来の `sonoisa/sentence-bert-base-ja-mean-tokens-v2` は新版 transformers と非互換のため使わない

## 注意

- Python は **3.12** で実行（`pip install -U transformers sentence-transformers fugashi unidic-lite` 済みの環境）。
- 3.8 系では古い tokenizers 0.12 が残っており動かない。

## 1. セットアップ — YAML / 会話データのロード

In [ ]:
from pathlib import Path
from collections import Counter
import re
import yaml
import numpy as np

ROOT = Path('.').resolve()
if (ROOT / 'protocol.yaml').exists():
    YAML_PATH = ROOT / 'protocol.yaml'
    DATASET_DIR = ROOT.parent / 'dataset'
else:
    YAML_PATH = ROOT / 'transition_diagram' / 'protocol.yaml'
    DATASET_DIR = ROOT / 'dataset'
TXT_PATH = DATASET_DIR / 'headache_emergency_calls.txt'

with YAML_PATH.open(encoding='utf-8') as f:
    data = yaml.safe_load(f)
protocols = {p['id']: p for p in data['protocols']}
proto_node_maps = {pid: {n['id']: n for n in p.get('nodes', [])} for pid, p in protocols.items()}

print(f'YAML: {YAML_PATH}')
print(f'TXT : {TXT_PATH}  exists={TXT_PATH.exists()}')
print(f'プロトコル数: {len(protocols)}')

## 2. 会話パーサ — `case_id` ブロックを発話列に分解

1 ケース = `case_id` / `triage_label` / `setting` / `turns: [(speaker, text), ...]`

In [ ]:
def parse_cases(text: str):
    cases = []
    pattern = re.compile(r'case_id:\s*(\S+).*?(?=case_id:|\Z)', re.S)
    for m in pattern.finditer(text):
        block = m.group(0)
        case_id = m.group(1).strip()
        triage_label = (re.search(r'triage_label:\s*(\S+)', block) or [None, None])[1]
        setting = (re.search(r'setting:\s*(.+)', block) or [None, None])[1]
        vec_match = re.search(r'vector:\s*(\[[^\]]+\])', block)
        try:
            vector = eval(vec_match.group(1)) if vec_match else None
        except Exception:
            vector = None
        turns = []
        for ln in block.splitlines():
            m2 = re.match(r'^(Dispatcher|Caller):\s*(.+)$', ln.strip())
            if m2:
                turns.append((m2.group(1), m2.group(2).strip()))
        cases.append({
            'case_id': case_id, 'triage_label': triage_label,
            'setting': setting, 'vector': vector, 'turns': turns,
        })
    return cases

cases = parse_cases(TXT_PATH.read_text(encoding='utf-8'))
print(f'読み込んだケース数: {len(cases)}')
print(f'triage_label 分布: {Counter(c["triage_label"] for c in cases)}')
print()
print(f'--- 例: {cases[0]["case_id"]} (label={cases[0]["triage_label"]}) ---')
for sp, t in cases[0]['turns']:
    print(f'  {sp:10s} {t}')

## 3. SBERT ロード

`intfloat/multilingual-e5-small` を採用。E5 系は
クエリ側に `'query: '`、文書側に `'passage: '` プレフィクスを付ける規約。

In [ ]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = 'intfloat/multilingual-e5-small'
print(f'Loading {MODEL_NAME} ...')
model = SentenceTransformer(MODEL_NAME)
print(f'embed dim: {model.get_sentence_embedding_dimension()}')

def encode_query(texts):
    return model.encode(['query: ' + t for t in texts], normalize_embeddings=True)

def encode_passage(texts):
    return model.encode(['passage: ' + t for t in texts], normalize_embeddings=True)

# 動作確認
qs = ['激しい痛みが、突然起こりましたか？']
us = ['その頭の痛みは、急に強く出たんですね。', '住所をお願いします。']
sim = encode_query(qs) @ encode_passage(us).T
print(f'類似度サンプル: {np.round(sim, 3).tolist()}')

## 4. パイプライン本体 — `walk_with_conversation`

プロトコルを 1 ノードずつ進め、各質問ごとに会話から choice を抽出する。

In [ ]:
TRIAGE_PRIORITY = {'R1': 6, 'R2': 5, 'R3': 4, 'Y1': 3, 'Y2': 2, 'G': 1}

def max_triage(triages):
    valid = [t for t in triages if t in TRIAGE_PRIORITY]
    return max(valid, key=lambda t: TRIAGE_PRIORITY[t]) if valid else None

def lookup_node(nid):
    for pid in protocols:
        if nid in proto_node_maps[pid]:
            return proto_node_maps[pid][nid]
    return None

def classify_choice_sbert(reply: str, question: str, choices):
    """reply に対し各 choice テンプレ「{質問} の答えは {choice_text}」との類似度を取り最大を返す"""
    if not reply:
        return 'c', {}
    templates = [f'{question} の答えは {c.get("text", c.get("value", ""))}' for c in choices]
    temp_emb = encode_passage(templates)
    rep_emb = encode_query([reply])[0]
    sims = temp_emb @ rep_emb
    best = int(np.argmax(sims))
    return choices[best].get('code'), {c.get('code'): float(s) for c, s in zip(choices, sims)}

def walk_with_conversation(protocol_id, turns, *, sim_threshold=0.75, verbose=False):
    proto = protocols[protocol_id]
    proto_nodes = proto_node_maps[protocol_id]
    fallback = proto.get('fallback', {})

    disp_idx = [i for i, (sp, _) in enumerate(turns) if sp == 'Dispatcher']
    disp_texts = [turns[i][1] for i in disp_idx]
    if not disp_texts:
        return {'final_triage': None, 'steps': []}
    disp_emb = encode_passage(disp_texts)

    current = proto.get('start_node') or proto.get('nodes', [{}])[0].get('id')
    steps, accumulated, orals, used = [], [], [], set()

    def finalize(terminal=None, route=None, fb=False):
        all_tri = list(accumulated) + ([terminal] if terminal else [])
        return {'steps': steps, 'accumulated_triages': accumulated,
                'terminal_triage': terminal, 'final_triage': max_triage(all_tri),
                'route_to_protocol': route, 'fallback_used': fb,
                'oral_instructions': orals}

    while current:
        node = proto_nodes.get(current) or lookup_node(current)
        if node is None:
            return finalize()
        if node.get('triage') and not node.get('choices'):
            orals.extend(node.get('oral_instruction') or [])
            steps.append({'node': current, 'type': 'node_terminal', 'tri': node['triage']})
            return finalize(terminal=node['triage'])
        if node.get('metadata_only') or node.get('transition_only'):
            current = node.get('next')
            continue
        choices = node.get('choices') or []
        if not choices:
            current = node.get('next')
            continue

        q_text = node.get('question', '')
        sims = (disp_emb @ encode_query([q_text])[0]).copy()
        for i in used:
            sims[i] = -1
        best_i = int(np.argmax(sims))
        best_sim = float(sims[best_i])

        if best_sim < sim_threshold:
            choice_code, best_disp, reply = 'c', '(no match)', ''
        else:
            used.add(best_i)
            best_disp = disp_texts[best_i]
            orig_idx = disp_idx[best_i]
            reply = next((turns[j][1] for j in range(orig_idx + 1, len(turns))
                          if turns[j][0] == 'Caller'), '')
            choice_code, _ = classify_choice_sbert(reply, q_text, choices)

        ch = next((c for c in choices if c.get('code') == choice_code), None) or choices[-1]
        ch_tri = ch.get('triage')
        orals.extend(ch.get('oral_instruction') or [])
        steps.append({
            'node': current, 'q': q_text, 'disp': best_disp,
            'sim': round(best_sim, 3), 'reply': reply,
            'choice': ch.get('code'), 'choice_text': ch.get('text'), 'tri': ch_tri,
        })
        if verbose:
            print(f'  [{current}] sim={best_sim:.3f}  reply="{reply[:30]}"  → {ch.get("code")} ({ch_tri})')

        if ch.get('route_to_protocol'):
            return finalize(terminal=ch_tri, route=ch['route_to_protocol'])
        nxt = ch.get('next')
        accumulate = bool(ch.get('accumulate_triage'))
        if nxt:
            if ch_tri and accumulate:
                accumulated.append(ch_tri)
                current = nxt
            elif ch_tri:
                return finalize(terminal=ch_tri)
            else:
                current = nxt
        elif ch_tri:
            return finalize(terminal=ch_tri)
        else:
            is_unknown = (ch.get('code') == 'c') or ('不明' in (ch.get('text') or ''))
            ftri = fallback.get('if_any_unknown') if is_unknown else \
                   fallback.get('if_all_symptom_questions_negative')
            return finalize(terminal=ftri, fb=True)
    return finalize()

## 5. デモ — 1 ケースを詳細トレース

In [ ]:
case = cases[0]
print(f'=== {case["case_id"]}  label={case["triage_label"]} ===')
print(f'setting: {case["setting"]}')
print('\n会話:')
for sp, t in case['turns']:
    print(f'  {sp:10s} {t}')
print('\nパイプライン:')
res = walk_with_conversation('headache', case['turns'], verbose=True)
print(f'\n→ 予測トリアージ: {res["final_triage"]}  /  正解: {case["triage_label"]}')
print(f'   詳細: terminal={res["terminal_triage"]}  accumulated={res["accumulated_triages"]}  fb={res["fallback_used"]}')

## 6. 全ケース評価 — accuracy と混同行列

In [ ]:
results = []
for c in cases:
    res = walk_with_conversation('headache', c['turns'])
    results.append({
        'case_id': c['case_id'],
        'label': c['triage_label'],
        'pred': res['final_triage'],
        'fb_used': res['fallback_used'],
        'steps': res['steps'],
    })

correct = sum(1 for r in results if r['label'] == r['pred'])
print(f'accuracy: {correct}/{len(results)} = {correct/len(results):.1%}')
print(f'label   分布: {dict(Counter(r["label"] for r in results))}')
print(f'predict 分布: {dict(Counter(r["pred"] for r in results))}')

# 混同行列
TRI = ['R1', 'R2', 'R3', 'Y1', 'Y2', 'G']
cm = {(l, p): 0 for l in TRI + ['?'] for p in TRI + ['?']}
for r in results:
    l = r['label'] if r['label'] in TRI else '?'
    p = r['pred'] if r['pred'] in TRI else '?'
    cm[(l, p)] += 1

print('\n混同行列 (行=label, 列=pred):')
header = '       ' + ' '.join(f'{p:>5s}' for p in TRI + ['?'])
print(header)
for l in TRI + ['?']:
    row = ' '.join(f'{cm[(l, p)]:>5d}' for p in TRI + ['?'])
    print(f'  {l:>3s} | {row}')

## 7. 誤判定の中身を見る

どの質問で choice が間違ったのか追跡する。

In [ ]:
mismatches = [r for r in results if r['label'] != r['pred']]
print(f'誤判定: {len(mismatches)} 件\n')

# label×pred 別に件数を見る
from collections import defaultdict
buckets = defaultdict(list)
for m in mismatches:
    buckets[(m['label'], m['pred'])].append(m)
for key, ms in sorted(buckets.items(), key=lambda x: -len(x[1])):
    print(f'  {key[0]} → {key[1]}: {len(ms)} 件  (例: {[m["case_id"] for m in ms[:3]]})')

print('\n--- 誤判定 詳細（最初の3件） ---')
for m in mismatches[:3]:
    print(f'\n### {m["case_id"]}  label={m["label"]}  pred={m["pred"]}')
    for s in m['steps']:
        if s.get('type') == 'node_terminal':
            print(f'  [{s["node"]}] => triage {s["tri"]}')
        else:
            print(f'  [{s["node"]}] sim={s["sim"]} reply="{s["reply"][:50]}" → {s["choice"]} ({s["tri"]})')

## 8. 予測結果を CSV へ書き出し（任意）

In [ ]:
import csv
OUT = Path('output')
OUT.mkdir(exist_ok=True)
csv_path = OUT / 'headache_predictions.csv'
with csv_path.open('w', encoding='utf-8-sig', newline='') as f:
    w = csv.writer(f)
    w.writerow(['case_id', 'label', 'pred', 'correct', 'fb_used', 'path'])
    for r in results:
        path = ' / '.join(f'{s.get("node")}={s.get("choice", "-")}' for s in r['steps'])
        w.writerow([r['case_id'], r['label'], r['pred'],
                    int(r['label'] == r['pred']), int(r['fb_used']), path])
print(f'saved: {csv_path.resolve()}')